### 대법원 판례 수집

In [1]:
import pandas as pd
import requests
import time
import os
import re

# 1. 설정 정보
INPUT_FILE = "대법원_판례_리스트.csv"   # 판례 목록 파일
SAVE_DIR = "대법원_판례_html_raw"            # 저장할 폴더명 (이름 변경)
BASE_URL_HOST = "http://www.law.go.kr"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"폴더 생성 완료: {SAVE_DIR}")

# 3. 파일 로드
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"목록 파일 로드 완료: {len(df)}건")
else:
    print(f"오류: '{INPUT_FILE}' 파일이 없습니다.")
    exit()

# 헤더 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def sanitize_filename(name):
    """파일명에 쓸 수 없는 특수문자 제거"""
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

print(f"--- 개별 판례 HTML 수집 시작 ---")

# 4. 순회 및 저장
for idx, row in df.iterrows():
    try:
        # 파일명으로 쓸 사건번호 가져오기
        case_no = row.get('사건번호')
        link_part = row.get('판례상세링크')

        # 필수 정보 없으면 패스
        if pd.isna(case_no) or pd.isna(link_part):
            continue

        # 파일명 정제
        safe_name = sanitize_filename(case_no)
        # 확장자를 .json -> .html 로 변경
        file_path = os.path.join(SAVE_DIR, f"{safe_name}.html")

        # [이어하기 기능] 이미 수집된 파일이면 건너뛰기
        if os.path.exists(file_path):
            # print(f"[{idx+1}] 이미 존재함: {safe_name}")
            continue

        # URL 생성
        # JSON 요청용 변환 로직 제거하고 있는 그대로 사용
        full_url = f"{BASE_URL_HOST}{link_part}"

        # API 요청
        response = requests.get(full_url, headers=headers, timeout=15)

        if response.status_code == 200:
            # HTML 텍스트 가져오기
            html_content = response.text
            
            # 내용이 너무 짧으면(오류 페이지 등) 체크 (선택 사항)
            if len(html_content) > 100:
                # HTML 파일로 저장
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(html_content)
                
                # 진행 상황 출력 (50건마다)
                if (idx + 1) % 50 == 0:
                    print(f"[{idx+1}/{len(df)}] 저장 완료: {safe_name}")
            else:
                print(f"[{idx+1}] 내용 부실 확인: {full_url}")

        else:
            print(f"[{idx+1}] 요청 실패 ({response.status_code}): {safe_name}")

        # 서버 부하 방지
        time.sleep(0.2)

    except Exception as e:
        print(f"[{idx+1}] 에러 발생: {e}")

print("\n작업이 완료되었습니다.")

폴더 생성 완료: 대법원_판례_html_raw
목록 파일 로드 완료: 7100건
--- 개별 판례 HTML 수집 시작 ---
[50/7100] 저장 완료: 대법원-2025-두-34005
[100/7100] 저장 완료: 2022두53204
[150/7100] 저장 완료: 2024도14039
[200/7100] 저장 완료: 대법원-2025-두-33049
[250/7100] 저장 완료: 2025도2059
[300/7100] 저장 완료: 2024도16984
[350/7100] 저장 완료: 2021다245528
[400/7100] 저장 완료: 2024도12209
[450/7100] 저장 완료: 2021다216957
[500/7100] 저장 완료: 2024두60336
[550/7100] 저장 완료: 2024다291522
[600/7100] 저장 완료: 대법원-2024-다-293023
[650/7100] 저장 완료: 2024도7642
[700/7100] 저장 완료: 2024두41038
[750/7100] 저장 완료: 2024다241152
[800/7100] 저장 완료: 2022다243550
[850/7100] 저장 완료: 2024다254820
[900/7100] 저장 완료: 대법원2024두44457
[950/7100] 저장 완료: 대법원-2024-두-41564
[1000/7100] 저장 완료: 2022다217728
[1050/7100] 저장 완료: 2021두47974
[1100/7100] 저장 완료: 2024도4686
[1150/7100] 저장 완료: 2023두54112
[1200/7100] 저장 완료: 2024다218572
[1250/7100] 저장 완료: 2022다290648
[1300/7100] 저장 완료: 2023다315414
[1350/7100] 저장 완료: 2019다253700
[1400/7100] 저장 완료: 2020다290569
[1450/7100] 저장 완료: 2023그828
[1500/7100] 저장 완료: 2023다288703
[1550/7100] 저장

### 하위법원 데이터 수집

In [6]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import os
import json
import re

# 1. 설정 정보
INPUT_FILE = "하위법원_판례_리스트.csv"
SAVE_DIR = "하위법원_판례"
BASE_URL_HOST = "http://www.law.go.kr"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 3. 파일명 정제 함수 (미리 정의)
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

# 4. 데이터 로드 및 전처리
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"전체 목록 로드: {len(df)}건")
    
    # (1) 대법원 필터링
    if '데이터출처명' in df.columns:
        df = df[df['데이터출처명'] == '대법원'].copy() # copy()로 경고 방지
        print(f"-> '대법원' 필터링 후: {len(df)}건")
    
    # (2) [핵심] 이미 저장된 파일 목록 확인 (확장자 제거)
    # 폴더 내의 .json 파일들의 이름(사건번호)을 미리 집합(set)으로 만듭니다.
    downloaded_files = {f.replace('.json', '') for f in os.listdir(SAVE_DIR) if f.endswith('.json')}
    print(f"-> 현재 폴더에 저장된 파일: {len(downloaded_files)}건")

    # (3) DataFrame에 '파일명' 컬럼 미리 생성 (비교를 위해)
    # 사건번호를 파일명 규칙대로 변환해서 새로운 열을 만듭니다.
    df['safe_name'] = df['사건번호'].apply(sanitize_filename)

    # (4) 이미 다운로드된 건 제외 (차집합)
    # safe_name이 downloaded_files에 없는 행만 남깁니다.
    target_df = df[~df['safe_name'].isin(downloaded_files)]
    
    remaining_count = len(target_df)
    print(f"===> [최종 다운로드 대상]: {remaining_count}건 (이미 완료된 {len(df) - remaining_count}건 제외)")

    if remaining_count == 0:
        print("모든 파일이 이미 다운로드되었습니다. 작업을 종료합니다.")
        exit()

else:
    print(f"오류: '{INPUT_FILE}' 파일이 없습니다.")
    exit()

# --- 세션 설정 (재시도 로직) ---
session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print(f"\n--- 남은 {remaining_count}건 수집 시작 ---")

# 5. 순회 및 저장 (target_df만 순회하므로 이미 받은 건 건너뜀)
# total 변수는 진행률 표시용
for count, (idx, row) in enumerate(target_df.iterrows(), start=1):
    try:
        case_no = row.get('사건번호')
        link_part = row.get('판례상세링크')
        safe_name = row.get('safe_name') # 위에서 만들어둔 컬럼 사용
        
        file_path = os.path.join(SAVE_DIR, f"{safe_name}.json")
        
        # URL 생성
        link_part_json = link_part.replace("type=HTML", "type=JSON")
        full_url = f"{BASE_URL_HOST}{link_part_json}"

        # 요청 및 에러 핸들링
        try:
            response = session.get(full_url, timeout=20)
            
            if response.status_code == 200:
                data = response.json()
                prec_service = data.get("PrecService", {})
                target_info = prec_service.get("Prec", prec_service)
                
                if target_info and ("판례일련번호" in target_info or "판례정보일련번호" in target_info):
                    with open(file_path, 'w', encoding='utf-8') as f:
                        json.dump(target_info, f, ensure_ascii=False, indent=4)
                    
                    # 진행 상황 출력 (현재 / 남은 건수)
                    if count % 10 == 0: # 10건마다 출력으로 변경 (확인 자주 하도록)
                        print(f"[{count}/{remaining_count}] 저장 완료: {safe_name}")
                else:
                    print(f"[{count}] 내용 없음: {full_url}")
            else:
                print(f"[{count}] 요청 실패 ({response.status_code}): {safe_name}")

            # 대기 시간 (1.2초로 약간 더 여유 있게)
            time.sleep(1.2)

        except requests.exceptions.RequestException as e:
            print(f"\n!!! [{count}/{remaining_count}] 차단/네트워크 에러: {e}")
            print("!!! 60초 대기 후 재시도합니다...")
            time.sleep(60)
            # 여기서는 continue하지 않고 다음 루프로 넘어가게 둡니다. 
            # (엄밀한 재시도 로직을 짜면 복잡해지니, 일단 1분 쉬고 다음 건으로 넘어갑니다)

    except Exception as e:
        print(f"[{count}] 일반 에러: {e}")

print("\n작업이 완료되었습니다.")

전체 목록 로드: 93721건
-> '대법원' 필터링 후: 27826건
-> 현재 폴더에 저장된 파일: 25798건
===> [최종 다운로드 대상]: 1962건 (이미 완료된 25864건 제외)

--- 남은 1962건 수집 시작 ---
[10/1962] 저장 완료: 73나2682
[20/1962] 저장 완료: 72구291
[30/1962] 저장 완료: 73코3
[40/1962] 저장 완료: 74노22
[50/1962] 저장 완료: 74노12
[60/1962] 저장 완료: 72구44
[70/1962] 저장 완료: 73나1359
[80/1962] 저장 완료: 72구32
[90/1962] 저장 완료: 73구383
[100/1962] 저장 완료: 73나1573
[110/1962] 저장 완료: 71구19
[120/1962] 저장 완료: 73나1061
[130/1962] 저장 완료: 73나2102
[140/1962] 저장 완료: 73나1093
[150/1962] 저장 완료: 73노1258
[160/1962] 저장 완료: 73나289
[170/1962] 저장 완료: 73나88
[180/1962] 저장 완료: 73나1005
[190/1962] 저장 완료: 73나775
[200/1962] 저장 완료: 71노939
[210/1962] 저장 완료: 73노838
[220/1962] 저장 완료: 72나2964
[230/1962] 저장 완료: 69나1143
[240/1962] 저장 완료: 73나17
[250/1962] 저장 완료: 72나2540
[260/1962] 저장 완료: 73노252
[270/1962] 저장 완료: 71나271
[280/1962] 저장 완료: 72나2867
[290/1962] 저장 완료: 72나1433
[300/1962] 저장 완료: 72나2262
[310/1962] 저장 완료: 72나2106
[320/1962] 저장 완료: 72나2422
[330/1962] 저장 완료: 72노1286
[340/1962] 저장 완료: 72나2066
[350/1962] 저장 완료:

### 하위법원 데이터 수집

In [2]:
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import os
import re

# 1. 설정 정보
INPUT_FILE = "하위법원_판례_리스트.csv"
SAVE_DIR = "하위법원_판례_html"      # 폴더명 변경 (html 구분)
BASE_URL_HOST = "http://www.law.go.kr"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"폴더 생성 완료: {SAVE_DIR}")

# 3. 파일명 정제 함수
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name)).strip()

# 4. 데이터 로드 및 전처리
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"전체 목록 로드: {len(df)}건")
    
    # [수정 1] '데이터출처명' 필터링 로직 제거 (전체 수집)
    # if '데이터출처명' in df.columns:
    #     df = df[df['데이터출처명'] == '대법원'].copy()
    
    # [수정 2] 이미 저장된 파일 목록 확인 (확장자 .html 기준)
    downloaded_files = {f.replace('.html', '') for f in os.listdir(SAVE_DIR) if f.endswith('.html')}
    print(f"-> 현재 폴더에 저장된 파일: {len(downloaded_files)}건")

    # DataFrame에 'safe_name' 컬럼 생성
    df['safe_name'] = df['사건번호'].apply(sanitize_filename)

    # 이미 다운로드된 건 제외 (차집합)
    target_df = df[~df['safe_name'].isin(downloaded_files)]
    
    remaining_count = len(target_df)
    print(f"===> [최종 다운로드 대상]: {remaining_count}건 (이미 완료된 {len(df) - remaining_count}건 제외)")

    if remaining_count == 0:
        print("모든 파일이 이미 다운로드되었습니다. 작업을 종료합니다.")
        exit()

else:
    print(f"오류: '{INPUT_FILE}' 파일이 없습니다.")
    exit()

# --- 세션 설정 (재시도 로직 유지) ---
session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print(f"\n--- 남은 {remaining_count}건 HTML 수집 시작 ---")

# 5. 순회 및 저장
for count, (idx, row) in enumerate(target_df.iterrows(), start=1):
    try:
        case_no = row.get('사건번호')
        link_part = row.get('판례상세링크')
        safe_name = row.get('safe_name')
        
        # [수정 3] 확장자 .html 로 변경
        file_path = os.path.join(SAVE_DIR, f"{safe_name}.html")
        
        # [수정 4] URL 생성 (JSON 변환 없이 그대로 사용)
        full_url = f"{BASE_URL_HOST}{link_part}"

        # 요청 및 에러 핸들링
        try:
            response = session.get(full_url, timeout=20)
            
            if response.status_code == 200:
                # [수정 5] 텍스트(HTML) 가져오기
                html_content = response.text
                
                # 내용 길이 체크 (빈 페이지 방지용, 기준은 임의로 100자)
                if len(html_content) > 100:
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(html_content)
                    
                    if count % 10 == 0: 
                        print(f"[{count}/{remaining_count}] 저장 완료: {safe_name}")
                else:
                    print(f"[{count}] 내용 부실 확인: {full_url}")
            else:
                print(f"[{count}] 요청 실패 ({response.status_code}): {safe_name}")

            # 대기 시간 (서버 보호 및 차단 방지)
            time.sleep(1.2)

        except requests.exceptions.RequestException as e:
            print(f"\n!!! [{count}/{remaining_count}] 차단/네트워크 에러: {e}")
            print("!!! 60초 대기 후 재시도합니다...")
            time.sleep(60)

    except Exception as e:
        print(f"[{count}] 일반 에러: {e}")

print("\n작업이 완료되었습니다.")

폴더 생성 완료: 하위법원_판례_html
전체 목록 로드: 94890건
-> 현재 폴더에 저장된 파일: 0건
===> [최종 다운로드 대상]: 94890건 (이미 완료된 0건 제외)

--- 남은 94890건 HTML 수집 시작 ---
[10/94890] 저장 완료: 수원지방법원-2025-가단-514391
[20/94890] 저장 완료: 안산지원-2025-가단-64793
[30/94890] 저장 완료: 인천지방법원-2024-구합-54335
[40/94890] 저장 완료: 서울남부지방법원-2024-가단-310915
[50/94890] 저장 완료: 울산지방법원-2023-구합-7196
[60/94890] 저장 완료: 서울고등법원-2024-나-2058747
[70/94890] 저장 완료: 안양지원-2025-가단-103284
[80/94890] 저장 완료: 서울고등법원-2024-나-2033892
[90/94890] 저장 완료: 수원지방법원-2023-구합-1331
[100/94890] 저장 완료: 서울행정법원-2021-구합-70813
[110/94890] 저장 완료: 서울고등법원-2025-누-3899
[120/94890] 저장 완료: 대구지방법원-2024-재나-30150
[130/94890] 저장 완료: 서울고등법원-2025-나-202392
[140/94890] 저장 완료: 2024나41282
[150/94890] 저장 완료: 춘천지방법원-2025-나-30292
[160/94890] 저장 완료: 대구지방법원 서부지원-2024-가단-55560
[170/94890] 저장 완료: 2024가합52067
[180/94890] 저장 완료: 안양지원-2025-가단-102844
[190/94890] 저장 완료: 서울중앙지방법원2024가단5496528
[200/94890] 저장 완료: 서울고등법원-2024-누-60680
[210/94890] 저장 완료: 수원지방법원-2023-가합-22449
[220/94890] 저장 완료: 의정부지방법원-2024-구단-530
[230/94890] 저장 

### 대법원 데이터 전처리

### 대법원 데이터 전처리